In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 3: Smoke test — forward pass + loss + PINN na GPU                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import tensorflow as tf
import numpy as np
from geosteering_ai.config import PipelineConfig
from geosteering_ai.models.registry import build_model
from geosteering_ai.losses.factory import build_combined
from geosteering_ai.losses.pinns import (
    build_pinns_loss,
    make_smoothness_loss,
    make_skin_depth_loss,
    make_variational_loss,
    make_self_adaptive_loss,
    make_continuity_loss,
    VALID_PINNS_SCENARIOS,
)

print("=" * 70)
print("SMOKE TEST: FORWARD PASS + LOSS + PINNs NA GPU")
print("=" * 70)

# ── 1. Construir modelo ResNet-18 baseline ───────────────────────────────────
config = PipelineConfig.baseline()
model = build_model(config)
print(f"\n✓ Modelo: {config.model_type} ({model.count_params():,} parâmetros)")

# ── 2. Forward pass com dados sintéticos ─────────────────────────────────────
batch_size, seq_len, n_features = 4, 100, config.n_features
x = tf.random.normal((batch_size, seq_len, n_features))
with tf.device('/GPU:0'):
    y_pred = model(x, training=False)
print(f"✓ Forward pass: input {x.shape} → output {y_pred.shape}")
assert y_pred.shape == (batch_size, seq_len, config.output_channels)

# ── 3. Loss combinada ────────────────────────────────────────────────────────
loss_fn = build_combined(config)
y_true = tf.random.uniform((batch_size, seq_len, config.output_channels), 0.0, 3.0)
loss_val = loss_fn(y_true, y_pred)
print(f"✓ Loss combinada: {loss_val.numpy():.6f} (scalar)")

# ── 4. Testar todos os 8 cenários PINN ───────────────────────────────────────
print(f"\n--- PINNs: {len(VALID_PINNS_SCENARIOS)} cenários ---")
for scenario in sorted(VALID_PINNS_SCENARIOS):
    cfg = PipelineConfig(use_pinns=True, pinns_scenario=scenario)
    lam_var = tf.Variable(0.01, dtype=tf.float32)
    pinns_fn = build_pinns_loss(cfg, pinns_lambda_var=lam_var)
    with tf.device('/GPU:0'):
        pinn_loss = pinns_fn(y_true, y_pred)
    print(f"  ✓ {scenario:15s} → loss = {pinn_loss.numpy():.8f}")

# ── 5. Verificar gradientes fluem pelas PINNs ────────────────────────────────
print(f"\n--- Verificação de gradientes ---")
cfg_grad = PipelineConfig(
    use_pinns=True, pinns_scenario="variational",
    use_tiv_constraint=True, tiv_constraint_weight=0.1,
)
model_grad = build_model(cfg_grad)
pinns_fn_grad = build_pinns_loss(cfg_grad, pinns_lambda_var=tf.Variable(0.01))

with tf.GradientTape() as tape:
    y_p = model_grad(x, training=True)
    data_loss = tf.reduce_mean(tf.square(y_true - y_p))
    pinn_loss = pinns_fn_grad(y_true, y_p)
    total_loss = data_loss + pinn_loss

grads = tape.gradient(total_loss, model_grad.trainable_variables)
non_none = sum(1 for g in grads if g is not None)
print(f"  ✓ Gradientes: {non_none}/{len(model_grad.trainable_variables)} variáveis recebem gradiente")
print(f"  ✓ Total loss = {total_loss.numpy():.6f} (data={data_loss.numpy():.4f} + pinn={pinn_loss.numpy():.6f})")

# ── 6. Benchmark: tempo de forward pass ──────────────────────────────────────
print(f"\n--- Benchmark GPU ---")
import time
x_bench = tf.random.normal((8, 600, n_features))
# Warmup
_ = model(x_bench, training=False)
# Medir
start = time.time()
for _ in range(100):
    _ = model(x_bench, training=False)
elapsed = time.time() - start
print(f"  ✓ 100 forward passes (batch=8, seq=600): {elapsed:.2f}s ({elapsed*10:.1f}ms/batch)")

print(f"\n{'=' * 70}")
print("✓ SMOKE TEST COMPLETO — GPU FUNCIONANDO CORRETAMENTE")
print("=" * 70)

## Célula 3 — Smoke Test Rápido (Forward Pass + Loss)

Teste mínimo para validar que o pipeline end-to-end funciona na GPU.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 2: Execução por módulo (para debugging isolado)                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import subprocess
import sys
import os
import time

DRIVE_PATH = "/content/drive/MyDrive/Geosteering_AI"
GITHUB_PATH = "/content/geosteering-ai"
PROJECT_DIR = DRIVE_PATH if os.path.exists(DRIVE_PATH) else GITHUB_PATH
TESTS_DIR = os.path.join(PROJECT_DIR, "tests")

# ── Módulos de teste em ordem de prioridade (TF-heavy primeiro) ──────────────
TEST_MODULES = [
    ("test_pinns.py",         52, "8 cenários PINN + TIV + factory"),
    ("test_models.py",        70, "44 arquiteturas (forward pass)"),
    ("test_losses.py",        37, "26 losses + gradientes + combined"),
    ("test_noise.py",         39, "15 tipos noise TF + curriculum"),
    ("test_training.py",      29, "TrainingLoop + callbacks + N-Stage"),
    ("test_strategies_bc.py",  3, "Oversampling + curriculum rho"),
    ("test_inference.py",      2, "InferencePipeline + export"),
    ("test_config.py",         0, "PipelineConfig (CPU-only, referência)"),
    ("test_data_pipeline.py",  0, "DataPipeline shapes (CPU-only)"),
    ("test_boundaries.py",     0, "DTB boundaries (CPU-only)"),
]

print("=" * 70)
print("EXECUÇÃO POR MÓDULO")
print("=" * 70)

total_passed = 0
total_failed = 0
total_skipped = 0
results = []

for module, expected_tf, desc in TEST_MODULES:
    filepath = os.path.join(TESTS_DIR, module)
    if not os.path.exists(filepath):
        print(f"  ⚠ {module}: NÃO ENCONTRADO")
        continue

    start = time.time()
    r = subprocess.run(
        [sys.executable, "-m", "pytest", filepath, "-v", "--tb=short", "--no-header"],
        capture_output=True, text=True, timeout=180,
    )
    elapsed = time.time() - start

    # ── Parsear contagens ─────────────────────────────────────────────────
    last_line = r.stdout.strip().split("\n")[-1] if r.stdout.strip() else ""
    passed = int(last_line.split(" passed")[0].split()[-1]) if "passed" in last_line else 0
    failed = int(last_line.split(" failed")[0].split()[-1]) if "failed" in last_line else 0
    skipped = int(last_line.split(" skipped")[0].split()[-1]) if "skipped" in last_line else 0

    status = "✓" if failed == 0 else "✗"
    total_passed += passed
    total_failed += failed
    total_skipped += skipped
    results.append((module, passed, failed, skipped, elapsed, status))

    print(f"  {status} {module:30s} {passed:3d}P {failed:3d}F {skipped:3d}S  ({elapsed:.1f}s) — {desc}")

    # Se houve falha, mostrar detalhes
    if failed > 0:
        print(f"\n    --- FALHAS em {module} ---")
        for line in r.stdout.split("\n"):
            if "FAILED" in line or "ERROR" in line:
                print(f"    {line.strip()}")
        print()

# ── Sumário ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print(f"SUMÁRIO: {total_passed} passed, {total_failed} failed, {total_skipped} skipped")
print("=" * 70)
if total_failed == 0:
    print("✓ VALIDAÇÃO GPU COMPLETA — TODOS OS TESTES PASSARAM!")
    if total_skipped == 0:
        print("  (Nenhum teste skipped — cobertura 100%)")
else:
    print(f"✗ {total_failed} FALHAS DETECTADAS — revisar módulos acima.")

## Célula 2 — Execução por Módulo (Debugging)

Se a Célula 1 falhar, execute módulo por módulo para isolar o problema.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 1: Execução completa do test suite com GPU                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import subprocess
import sys
import os
import time

# ── Determinar diretório do projeto ──────────────────────────────────────────
DRIVE_PATH = "/content/drive/MyDrive/Geosteering_AI"
GITHUB_PATH = "/content/geosteering-ai"
PROJECT_DIR = DRIVE_PATH if os.path.exists(DRIVE_PATH) else GITHUB_PATH

print("=" * 70)
print(f"EXECUTANDO TEST SUITE COMPLETO")
print(f"Diretório: {PROJECT_DIR}")
print("=" * 70)

start = time.time()

# ── Executar pytest com relatório detalhado ──────────────────────────────────
result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        os.path.join(PROJECT_DIR, "tests"),
        "-v",
        "--tb=short",
        "-x",           # para na primeira falha (debugging)
        "--no-header",
    ],
    capture_output=True,
    text=True,
    timeout=600,  # 10 minutos máximo
)

elapsed = time.time() - start

# ── Exibir resultados ────────────────────────────────────────────────────────
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr[-1000:])

print(f"\n{'=' * 70}")
print(f"Tempo total: {elapsed:.1f}s")
print(f"Exit code: {result.returncode}")
if result.returncode == 0:
    print("✓ TODOS OS TESTES PASSARAM!")
else:
    print("✗ FALHAS DETECTADAS — revisar saída acima.")

## Célula 1 — Execução Completa do Test Suite

Executa **todos** os 592+ testes (incluindo os 232 TF-dependent).
Se algum teste falhar, o relatório detalhado será exibido.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 0: Verificação de GPU + Instalação do pacote                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import subprocess
import sys

# ── Verificar GPU ────────────────────────────────────────────────────────────
print("=" * 70)
print("VERIFICAÇÃO DE GPU")
print("=" * 70)

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print(f"TensorFlow versão: {tf.__version__}")
    print(f"GPUs disponíveis:  {len(gpus)}")
    for gpu in gpus:
        print(f"  → {gpu.name} ({gpu.device_type})")
    if not gpus:
        print("\n⚠️  NENHUMA GPU DETECTADA!")
        print("    Vá em Runtime → Change runtime type → GPU")
        raise SystemExit("GPU necessária para validação.")
    # Configurar crescimento de memória para evitar OOM
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"\n✓ GPU configurada com memory growth habilitado")
except ImportError:
    print("ERRO: TensorFlow não instalado!")
    sys.exit(1)

# ── Instalar pacote Geosteering AI ──────────────────────────────────────────
print("\n" + "=" * 70)
print("INSTALAÇÃO DO PACOTE")
print("=" * 70)

# Opção A: via GitHub (recomendado para CI)
# !pip install -q "git+https://github.com/daniel-leal/geosteering-ai.git@main#egg=geosteering-ai[all]"

# Opção B: via Google Drive (recomendado para desenvolvimento)
import os
DRIVE_PATH = "/content/drive/MyDrive/Geosteering_AI"
GITHUB_PATH = "/content/geosteering-ai"

if os.path.exists(DRIVE_PATH):
    print(f"Usando pacote do Drive: {DRIVE_PATH}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{DRIVE_PATH}[all]"])
elif os.path.exists(GITHUB_PATH):
    print(f"Usando clone local: {GITHUB_PATH}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{GITHUB_PATH}[all]"])
else:
    print("Clonando do GitHub...")
    subprocess.check_call(["git", "clone", "-q", "https://github.com/daniel-leal/geosteering-ai.git", GITHUB_PATH])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{GITHUB_PATH}[all]"])

# ── Instalar pytest ──────────────────────────────────────────────────────────
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytest>=7.0", "pytest-cov>=4.0"])

# ── Verificar imports ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("VERIFICAÇÃO DE IMPORTS")
print("=" * 70)
from geosteering_ai.config import PipelineConfig
from geosteering_ai.models.registry import list_available_models
print(f"PipelineConfig: ✓ ({len(PipelineConfig.__dataclass_fields__)} campos)")
print(f"Modelos disponíveis: ✓ ({len(list_available_models())} arquiteturas)")
print(f"TensorFlow: ✓ ({tf.__version__})")
print(f"NumPy: ✓ ({__import__('numpy').__version__})")
print("\n✓ Ambiente pronto para validação.")

## Célula 0 — Verificação de GPU e Instalação

# Geosteering AI v2.0 — Validação GPU (Google Colab)

**Propósito:** Executar os 232 testes TF-skipped que requerem GPU/TensorFlow.

**Requisitos:**
- Runtime: GPU (T4 ou superior)
- RAM: 12.7 GB (padrão Colab)
- Disco: ~2 GB para pacotes

**Módulos testados:**
| Módulo | Testes TF | O que valida |
|:-------|:---------:|:-------------|
| `test_models.py` | 70 | Forward pass das 44 arquiteturas |
| `test_pinns.py` | 52 | 8 cenários PINN + TIV + build_pinns_loss |
| `test_noise.py` | 39 | Noise on-the-fly TF (15 tipos + curriculum) |
| `test_losses.py` | 37 | 26 losses + gradientes + combined |
| `test_training.py` | 29 | TrainingLoop, callbacks, N-Stage |
| `test_strategies_bc.py` | 3 | Oversampling + curriculum rho |
| `test_inference.py` | 2 | InferencePipeline + export |

**Autor:** Daniel Leal | **Versão:** v2.0.0 | **Data:** 2026-04-01